### Import the Basic Libraries 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

--- Or import the libraries from mine py ---

In [2]:
import sys
sys.path.append(r'C:\Users\User\005_Libraries')

import mylibs

In [3]:
import requests
import urllib.parse
import json
import time
from datetime import datetime
import os
from dotenv import load_dotenv
load_dotenv()

### Set Up the API Key

In [4]:
API_KEY= os.getenv('RIOT_API_KEY')
REGION= os.getenv('RIOT_REGION', 'euw1')

BASE_URL= f"https://{REGION}.api.riotgames.com"
EUROPE_URL= "https://europe.api.riotgames.com"

HEADERS= {"X-Riot-Token": API_KEY}

### Call & Test API

In [5]:
def get_player_data(game_name, tag_line):
    """
    Get complete player data using Riot ID (GameName#TagLine)
    
    Parameters:
    - game_name: Your game name (e.g., "MpampoKilleRR")
    - tag_line: Your tag (e.g., "EUW")
    
    Returns:
    - Dictionary with all player data
    """
    print(f"\n{'='*60}")
    print(f"Looking up player: {game_name}#{tag_line}")
    print('='*60)
    
    account_url= f"{EUROPE_URL}/riot/account/v1/accounts/by-riot-id/{game_name}/{tag_line}"
    
    try:
        account_response= requests.get(account_url, headers= HEADERS)
        
        if account_response.status_code== 200:
            account_data= account_response.json()
            print(f"Account found: {account_data['gameName']}#{account_data['tagLine']}")
        else:
            print(f"Account not found: {account_response.status_code}")
            return None
    
    except Exception as e:
        print(f"Error getting account: {e}")
        return None
    
    puuid= account_data['puuid']
    summoner_url= f"{BASE_URL}/lol/summoner/v4/summoners/by-puuid/{puuid}"
    
    try:
        summoner_response= requests.get(summoner_url, headers= HEADERS)
        
        if summoner_response.status_code== 200:
            summoner_data= summoner_response.json()
            print(f"Summoner data retrieved!")
        else:
            print(f"Failed to get summoner data: {summoner_response.status_code}")
            return None
            
    except Exception as e:
        print(f"Error getting summoner: {e}")
        return None
    
    player_data= {
        'gameName': account_data['gameName'],
        'tagLine': account_data['tagLine'],
        'puuid': account_data['puuid'],
        'summonerLevel': summoner_data['summonerLevel'],
        'profileIconId': summoner_data['profileIconId'],
        'revisionDate': summoner_data.get('revisionDate')
    }
    
    print(f"\n Player Summary:")
    print(f"   Player: {player_data['gameName']}#{player_data['tagLine']}")
    print(f"   Level: {player_data['summonerLevel']}")
    print(f"   Profile Icon: {player_data['profileIconId']}")
    print(f"   PUUID: {player_data['puuid']}")
    
    return player_data

get_player_data("MpampoKilleRR", "EUW")

In [6]:
my_player= get_player_data("MpampoKilleRR", "EUW")

if my_player:
    print("\n" + "="*60)
    print("YOUR LEAGUE ACCOUNT DATA:")
    print("="*60)
    for key, value in my_player.items():
        print(f"   {key}: {value}")


Looking up player: MpampoKilleRR#EUW
Account found: MpampoKilleRR#EUW
Summoner data retrieved!

 Player Summary:
   Player: MpampoKilleRR#EUW
   Level: 572
   Profile Icon: 3154
   PUUID: oLWAAwzBS-sFaPD8vW08OW8QNunhnYr1p6JLG5cc_lI9ub8j3Hy3flrFEyMsYMNdFLMbbMHGn6rzFg

YOUR LEAGUE ACCOUNT DATA:
   gameName: MpampoKilleRR
   tagLine: EUW
   puuid: oLWAAwzBS-sFaPD8vW08OW8QNunhnYr1p6JLG5cc_lI9ub8j3Hy3flrFEyMsYMNdFLMbbMHGn6rzFg
   summonerLevel: 572
   profileIconId: 3154
   revisionDate: 1762357358549


### Fetch Match History

In [7]:
def get_match_history(puuid, count= 20):
    """
    Get list of recent match IDs for a player
    
    Parameters:
    - puuid: Player's PUUID (the permanent unique ID)
    - count: Number of matches to retrieve (default 20, max 100)
    
    Returns:
    - List of match IDs
    """
    print(f"\n{'='*60}")
    print(f"Fetching last {count} matches")
    print('='*60)
    
    matches_url= f"{EUROPE_URL}/lol/match/v5/matches/by-puuid/{puuid}/ids"
    
    params= {
        'start': 0,
        'count': count
    }
    
    try:
        response= requests.get(matches_url, headers= HEADERS, params= params)
        
        if response.status_code== 200:
            match_ids= response.json()
            print(f"Retrieved {len(match_ids)} matches!")
            
            print(f"\n First 3 match IDs:")
            for i, match_id in enumerate(match_ids[:3], 1):
                print(f"   {i}. {match_id}")
            print(f"   ... and {len(match_ids) - 3} more")
            
            return match_ids
        else:
            print(f"Failed to get matches: {response.status_code}")
            print(f"Response: {response.text}")
            return None
            
    except Exception as e:
        print(f"Error: {e}")
        return None

In [8]:
get_match_history(my_player['puuid'], count= 20)


Fetching last 20 matches
Retrieved 20 matches!

 First 3 match IDs:
   1. EUW1_7593052094
   2. EUW1_7593018081
   3. EUW1_7592984954
   ... and 17 more


['EUW1_7593052094',
 'EUW1_7593018081',
 'EUW1_7592984954',
 'EUW1_7592974313',
 'EUW1_7592940679',
 'EUW1_7591494778',
 'EUW1_7591410390',
 'EUW1_7591371172',
 'EUW1_7591305145',
 'EUW1_7591268045',
 'EUW1_7591207997',
 'EUW1_7591163633',
 'EUW1_7591106683',
 'EUW1_7591069318',
 'EUW1_7591046441',
 'EUW1_7590766123',
 'EUW1_7590765600',
 'EUW1_7590757848',
 'EUW1_7590743220',
 'EUW1_7589977125']

In [9]:
if my_player:
    my_matches= get_match_history(my_player['puuid'], count= 20)
    
    if my_matches:
        print(f"\n{'='*60}")
        print(f"YOUR RECENT MATCHES:")
        print('='*60)
        print(f"Total matches retrieved: {len(my_matches)}")
        
        print(f"\n All Match IDs:")
        for i, match_id in enumerate(my_matches, 1):
            print(f"   {i}. {match_id}")


Fetching last 20 matches
Retrieved 20 matches!

 First 3 match IDs:
   1. EUW1_7593052094
   2. EUW1_7593018081
   3. EUW1_7592984954
   ... and 17 more

YOUR RECENT MATCHES:
Total matches retrieved: 20

 All Match IDs:
   1. EUW1_7593052094
   2. EUW1_7593018081
   3. EUW1_7592984954
   4. EUW1_7592974313
   5. EUW1_7592940679
   6. EUW1_7591494778
   7. EUW1_7591410390
   8. EUW1_7591371172
   9. EUW1_7591305145
   10. EUW1_7591268045
   11. EUW1_7591207997
   12. EUW1_7591163633
   13. EUW1_7591106683
   14. EUW1_7591069318
   15. EUW1_7591046441
   16. EUW1_7590766123
   17. EUW1_7590765600
   18. EUW1_7590757848
   19. EUW1_7590743220
   20. EUW1_7589977125


In [10]:
def count_total_matches(puuid, max_check= 10000):
    """
    Estimate total number of available matches
    (API doesn't give us a total count directly)
    """
    print(f"\n{'='*60}")
    print(f"Checking total matches available...")
    print('='*60)
    
    matches_url= f"{EUROPE_URL}/lol/match/v5/matches/by-puuid/{puuid}/ids"
    
    all_matches= []
    start= 0
    count= 100
    
    while start < max_check:
        params= {
            'start': start,
            'count': count
        }
        
        try:
            response= requests.get(matches_url, headers= HEADERS, params= params)
            
            if response.status_code== 200:
                match_batch= response.json()
                
                if len(match_batch)== 0:
                    break
                
                all_matches.extend(match_batch)
                print(f"   Retrieved {len(match_batch)} matches (Total so far: {len(all_matches)})")
                
                start += count
                time.sleep(0.5)
            else:
                print(f"Error: {response.status_code}")
                break
                
        except Exception as e:
            print(f"Error: {e}")
            break
    
    print(f"\n Total matches found: {len(all_matches)}")
    return all_matches

In [11]:
count_total_matches(my_player['puuid'], max_check= 10000)


Checking total matches available...
   Retrieved 100 matches (Total so far: 100)
   Retrieved 100 matches (Total so far: 200)
   Retrieved 100 matches (Total so far: 300)
   Retrieved 100 matches (Total so far: 400)
   Retrieved 100 matches (Total so far: 500)
   Retrieved 100 matches (Total so far: 600)
   Retrieved 100 matches (Total so far: 700)
   Retrieved 100 matches (Total so far: 800)
   Retrieved 100 matches (Total so far: 900)
   Retrieved 100 matches (Total so far: 1000)

 Total matches found: 1000


['EUW1_7593052094',
 'EUW1_7593018081',
 'EUW1_7592984954',
 'EUW1_7592974313',
 'EUW1_7592940679',
 'EUW1_7591494778',
 'EUW1_7591410390',
 'EUW1_7591371172',
 'EUW1_7591305145',
 'EUW1_7591268045',
 'EUW1_7591207997',
 'EUW1_7591163633',
 'EUW1_7591106683',
 'EUW1_7591069318',
 'EUW1_7591046441',
 'EUW1_7590766123',
 'EUW1_7590765600',
 'EUW1_7590757848',
 'EUW1_7590743220',
 'EUW1_7589977125',
 'EUW1_7589919370',
 'EUW1_7589865839',
 'EUW1_7589814690',
 'EUW1_7589802026',
 'EUW1_7589760043',
 'EUW1_7588716574',
 'EUW1_7588702834',
 'EUW1_7588657406',
 'EUW1_7588607080',
 'EUW1_7588570767',
 'EUW1_7588532797',
 'EUW1_7588504661',
 'EUW1_7588474461',
 'EUW1_7588453814',
 'EUW1_7588435405',
 'EUW1_7588417932',
 'EUW1_7587853762',
 'EUW1_7587821043',
 'EUW1_7587762767',
 'EUW1_7587050804',
 'EUW1_7586985824',
 'EUW1_7586901375',
 'EUW1_7586854605',
 'EUW1_7586787344',
 'EUW1_7585514252',
 'EUW1_7585481234',
 'EUW1_7585436623',
 'EUW1_7584426849',
 'EUW1_7583720465',
 'EUW1_7583623822',


### Get Detailed Match Data

In [12]:
def get_match_details(match_id):
    """
    Get detailed information about a specific match
    
    Parameters:
    - match_id: The match ID (e.g., 'EUW1_7583720465')
    
    Returns:
    - Dictionary with all match data
    """
    match_url= f"{EUROPE_URL}/lol/match/v5/matches/{match_id}"
    
    try:
        response= requests.get(match_url, headers= HEADERS)
        
        if response.status_code== 200:
            match_data= response.json()
            return match_data
        else:
            print(f"Failed to get match {match_id}: {response.status_code}")
            return None
            
    except Exception as e:
        print(f"Error getting match {match_id}: {e}")
        return None

In [13]:
get_match_details('EUW1_7583720465')

{'metadata': {'dataVersion': '2',
  'matchId': 'EUW1_7583720465',
  'participants': ['vofiW22oiRqdOGtLouxTF4no4Er1w-FqF-1VkYpF3kFJRlcCwQaAlrpBVXyUzNHRztjwtENkzLb21g',
   '3EjyYMjqVNIMtX1g5DWytiL_h7suo8BU8dr8esIEXf3wSFVv7m-CQphLaS5DONcSosnfeO7g0Lh8_A',
   'oLWAAwzBS-sFaPD8vW08OW8QNunhnYr1p6JLG5cc_lI9ub8j3Hy3flrFEyMsYMNdFLMbbMHGn6rzFg',
   'k68yzq0-coTNnP58P5MKfJP-btYyQmuRegax-jBcMXGLJrb1NtJ0gVxmm9GDu1dTrpstdEScrvBaQw',
   'GqgvWQx6wD8H5P_afgygMDH7ekRx_wTdjmBHJsBfl4_RiNbJVCZHf6Y-AXIsguufINtOglw3lAps5w',
   'O0qG7ZRoZPEyl1qW0_vMEnEoIWhLoC1BkBsjgrBm9RH7pBEaoIBUozC1RMXNFB35tUMEQvUPFmlvCQ',
   '5U9rerW5VsPyYF1TjyoGlUvtBEq70PyXFv3Hvnwa83fZlUg6GKXsu7HViABvEOhhLvr9aqePvMAzMA',
   'xogt4y29oycVhqO9WdZNyscpWuulA5br2Ch-4DdSpVbpnnSjf3Dwwzl4u3TQjPulpX1j1jZYhZw_0Q',
   '9WDCPgykDGG6yL0Oxe12AVmR7LTPpYs-RF9Ldxk_uTXYpZ6fljYUCYkqfDRiFKZzjVocV_4ZdFAVLA',
   'LAPBLpHQgENs9aTGGyLVHwP1ndG6ZhtsfSD6L-W5Obe3-8OdjNvEhmiEfFE-Z5XdqZWV-9zPBV4UkQ']},
 'info': {'endOfGameResult': 'GameComplete',
  'gameCreation': 176

In [14]:
if my_matches:
    print(f"\n{'='*60}")
    print(f"GETTING DETAILED DATA FOR FIRST MATCH")
    print('='*60)
    
    first_match_id= my_matches[0]
    print(f"Match ID: {first_match_id}")
    
    match_details = get_match_details(first_match_id)
    
    if match_details:
        print(f"\n Match data retrieved!")
        
        print(f"\nMatch Data Structure:")
        print(f"Top-level keys: {list(match_details.keys())}")
        
        info= match_details['metadata']
        print(f"\n Match Metadata:")
        print(f"   Match ID: {info['matchId']}")
        print(f"   Participants: {len(info['participants'])} players")
        
        game_info= match_details['info']
        print(f"\n Game Info:")
        print(f"   Game Mode: {game_info['gameMode']}")
        print(f"   Game Duration: {game_info['gameDuration'] // 60} minutes {game_info['gameDuration'] % 60} seconds")
        print(f"   Game Version: {game_info['gameVersion']}")
        
        participants= game_info['participants']
        your_puuid= my_player['puuid']
        
        for player in participants:
            if player['puuid']== your_puuid:
                print(f"\n YOUR PERFORMANCE:")
                print(f"   Champion: {player['championName']}")
                print(f"   KDA: {player['kills']}/{player['deaths']}/{player['assists']}")
                print(f"   Win: {'Victory' if player['win'] else 'Defeat'}")
                print(f"   Gold Earned: {player['goldEarned']:,}")
                print(f"   Total Damage to Champions: {player['totalDamageDealtToChampions']:,}")
                break


GETTING DETAILED DATA FOR FIRST MATCH
Match ID: EUW1_7593052094

 Match data retrieved!

Match Data Structure:
Top-level keys: ['metadata', 'info']

 Match Metadata:
   Match ID: EUW1_7593052094
   Participants: 10 players

 Game Info:
   Game Mode: CLASSIC
   Game Duration: 31 minutes 21 seconds
   Game Version: 15.22.723.1955

 YOUR PERFORMANCE:
   Champion: Kalista
   KDA: 5/2/7
   Win: Victory
   Gold Earned: 13,406
   Total Damage to Champions: 18,949


### Test With Katarina + CLASSIC

In [15]:
def count_katarina_games(puuid, total_matches= 100):
    """
    Count how many Katarina games in CLASSIC mode we have
    
    Parameters:
    - puuid: My PUUID
    - total_matches: How many matches to check (default 100)
    
    Returns:
    - Number of Katarina CLASSIC games found
    """
    print(f"\n{'='*60}")
    print(f"Analyzing my matches for Katarina games...")
    print('='*60)
    
    matches_url= f"{EUROPE_URL}/lol/match/v5/matches/by-puuid/{puuid}/ids"
    params= {
            'start': 0, 
            'count': total_matches
            }
    
    try:
        response= requests.get(matches_url, headers= HEADERS, params= params)
        if response.status_code != 200:
            print(f"Error getting matches: {response.status_code}")
            return 0
        
        match_ids= response.json()
        print(f"Retrieved {len(match_ids)} match IDs")
        
    except Exception as e:
        print(f"Error: {e}")
        return 0
    
    katarina_count= 0
    other_modes_count= 0
    other_champs_count= 0
    
    print(f"\n Checking each match...")
    
    for i, match_id in enumerate(match_ids):
        match_data= get_match_details(match_id)
        
        if not match_data:
            continue
        
        game_info= match_data['info']
        game_mode= game_info['gameMode']
        
        for player in game_info['participants']:
            if player['puuid']== puuid:
                champion= player['championName']
                
                if champion== 'Katarina' and game_mode== 'CLASSIC':
                    katarina_count += 1
                    print(f"Match {i+1}/{len(match_ids)}: Katarina CLASSIC (Total: {katarina_count})")
                elif champion== 'Katarina' and game_mode != 'CLASSIC':
                    other_modes_count += 1
                    print(f"Match {i+1}/{len(match_ids)}: Katarina {game_mode}")
                else:
                    other_champs_count += 1
                    print(f"Match {i+1}/{len(match_ids)}: {champion} {game_mode}")
                break
        
        time.sleep(0.6)
    
    print(f"\n {'='*60}")
    print(f"RESULTS:")
    print('='*60)
    print(f"   Katarina CLASSIC games: {katarina_count}")
    print(f"   Katarina other modes: {other_modes_count}")
    print(f"   Other champions: {other_champs_count}")
    print(f"   Total checked: {len(match_ids)}")
    
    return katarina_count

### Test With my First 100 Matches

In [16]:
katarina_games= count_katarina_games(my_player['puuid'], total_matches= 100)

print(f"\n Found {katarina_games} Katarina CLASSIC games in my last 100 matches!")


Analyzing my matches for Katarina games...
Retrieved 100 match IDs

 Checking each match...
Match 1/100: Kalista CLASSIC
Match 2/100: Katarina CLASSIC (Total: 1)
Match 3/100: Katarina CLASSIC (Total: 2)
Match 4/100: Jinx CLASSIC
Match 5/100: Yunara CLASSIC
Match 6/100: Katarina CLASSIC (Total: 3)
Match 7/100: Senna CLASSIC
Match 8/100: Katarina CLASSIC (Total: 4)
Match 9/100: Jinx CLASSIC
Match 10/100: Jinx CLASSIC
Match 11/100: Katarina CLASSIC (Total: 5)
Match 12/100: Senna CLASSIC
Match 13/100: Smolder CLASSIC
Match 14/100: Katarina CLASSIC (Total: 6)
Match 15/100: Katarina CLASSIC (Total: 7)
Match 16/100: Katarina CLASSIC (Total: 8)
Match 17/100: Katarina CLASSIC (Total: 9)
Match 18/100: Smolder CLASSIC
Match 19/100: Smolder CLASSIC
Match 20/100: Yone CLASSIC
Match 21/100: Katarina CLASSIC (Total: 10)
Match 22/100: Kayle CLASSIC
Match 23/100: Smolder CLASSIC
Match 24/100: Katarina CLASSIC (Total: 11)
Match 25/100: Smolder CLASSIC
Match 26/100: Katarina CLASSIC (Total: 12)
Match 27

### Data Collection Pipeline

In [17]:
def extract_katarina_match_data(match_data, your_puuid):
    """
    Extract all relevant features (21) from Katarina CLASSIC match
    
    Parameters:
    - match_data: Full match data from API
    - your_puuid: Your PUUID to identify your stats
    
    Returns:
    - Dictionary with all features, or None if not Katarina CLASSIC
    """
    game_info= match_data['info']
    
    # Check if it's CLASSIC mode
    if game_info['gameMode'] != 'CLASSIC':
        return None
    
    # Find your stats and your team
    your_stats= None
    your_team_id= None
    
    # Check if you played Katarina
    for player in game_info['participants']:
        if player['puuid'] == your_puuid:
            if player['championName'] != 'Katarina':
                return None
            
            your_stats= player
            your_team_id= player['teamId']
            break
    
    if not your_stats:
        return None
    
    # Calculate team totals (kills and deaths)
    team_kills= 0
    team_deaths= 0
    
    for player in game_info['participants']:
        if player['teamId'] == your_team_id:
            team_kills += player['kills']
            team_deaths += player['deaths']
    
    # Get team objectives
    team_objectives = None
    for team in game_info['teams']:
        if team['teamId'] == your_team_id:
            team_objectives = team['objectives']
            break
    
    # Extract game duration (in seconds)
    game_duration= game_info['gameDuration']
    game_duration_min= game_duration / 60
    
    # Extract YOUR performance stats
    kills= your_stats['kills']
    deaths= your_stats['deaths']
    assists= your_stats['assists']
    cs= your_stats['totalMinionsKilled'] + your_stats.get('neutralMinionsKilled', 0)
    gold_earned= your_stats['goldEarned']
    damage_to_champions= your_stats['totalDamageDealtToChampions']
    damage_taken= your_stats['totalDamageTaken']
    level= your_stats['champLevel']
    win= your_stats['win']
    
    # Extract team objectives
    team_dragons= team_objectives['dragon']['kills']
    team_barons= team_objectives['baron']['kills']
    team_towers= team_objectives['tower']['kills']
    first_blood= team_objectives['champion']['first']
    
    # Calculated features
    # KDA ratio (handle deaths= 0)
    if deaths == 0:
        kda_ratio= kills + assists 
    else:
        kda_ratio= (kills + assists) / deaths
    
    # Per minute stats
    cs_per_min= cs / game_duration_min
    gold_per_min= gold_earned / game_duration_min
    damage_per_min= damage_to_champions / game_duration_min
    
    # Participation and share stats
    # Kill participation (handle team_kills= 0)
    if team_kills == 0:
        kill_participation= 0
        kill_share= 0
    else:
        kill_participation= (kills + assists) / team_kills
        kill_share= kills / team_kills
    
    # Death share (handle team_deaths= 0)
    if team_deaths == 0:
        death_share= 0
    else:
        death_share= deaths / team_deaths
    
    # Return all features as a dictionary
    return {
        # Match info
        'game_duration': game_duration,
        'win': 1 if win else 0,
        
        # Your performance
        'kills': kills,
        'deaths': deaths,
        'assists': assists,
        'cs': cs,
        'gold_earned': gold_earned,
        'damage_to_champions': damage_to_champions,
        'damage_taken': damage_taken,
        'level': level,
        
        # Team performance
        'team_dragons': team_dragons,
        'team_barons': team_barons,
        'team_towers': team_towers,
        'first_blood': 1 if first_blood else 0,
        
        # Calculated features
        'kda_ratio': round(kda_ratio, 2),
        'cs_per_min': round(cs_per_min, 2),
        'gold_per_min': round(gold_per_min, 2),
        'damage_per_min': round(damage_per_min, 2),
        'kill_participation': round(kill_participation, 3),
        'kill_share': round(kill_share, 3),
        'death_share': round(death_share, 3)
    }

### Data Collection Function

In [18]:
def collect_katarina_dataset(puuid, num_matches= 1000):
    """
    Collect complete dataset of Katarina CLASSIC matches
    Handles API limit of 100 matches per request by batching
    
    Parameters:
    - puuid: Your PUUID
    - num_matches: Total number of matches to check (will batch in groups of 100)
    
    Returns:
    - pandas DataFrame with all Katarina match data
    """
    print(f"\n {'='*60}")
    print(f"COLLECTING KATARINA DATASET")
    print('='*60)
    print(f"Target: Check last {num_matches} matches...")
    
    # API limit is 100 matches per request
    batch_size= 100
    all_match_ids= []
    
    # Calculate number of batches needed
    num_batches= (num_matches + batch_size - 1) // batch_size
    
    print(f"Will make {num_batches} API requests (batches of {batch_size})\n")
    
    # Get match IDs in batches
    for batch_num in range(num_batches):
        start= batch_num * batch_size
        count= min(batch_size, num_matches - start)
        
        matches_url= f"{EUROPE_URL}/lol/match/v5/matches/by-puuid/{puuid}/ids"
        params= {'start': start, 'count': count}
        
        print(f"Batch {batch_num + 1}/{num_batches}: Fetching matches {start} to {start + count - 1}...")
        
        try:
            response= requests.get(matches_url, headers= HEADERS, params= params)
            
            if response.status_code == 200:
                batch_ids= response.json()
                all_match_ids.extend(batch_ids)
                print(f"Retrieved {len(batch_ids)} match IDs (Total so far: {len(all_match_ids)})")
            else:
                print(f"Error {response.status_code}: {response.text}")
                break
                
        except Exception as e:
            print(f"Error: {e}")
            break
        
        # Rate limiting between batches
        if batch_num < num_batches - 1:
            time.sleep(1)
    
    if not all_match_ids:
        print("\n No match IDs retrieved!")
        return None
    
    print(f"\n Total match IDs retrieved: {len(all_match_ids)}")
    
    # Collect data for each match
    all_matches_data= []
    katarina_count= 0
    
    print(f"\n {'='*60}")
    print(f"PROCESSING MATCHES FOR KATARINA GAMES")
    print(f"{'='*60}\n")
    
    # Get match details
    for i, match_id in enumerate(all_match_ids):
        match_data= get_match_details(match_id)
        
        if not match_data:
            print(f"Match {i+1}/{len(all_match_ids)}: Failed to retrieve")
            continue
        
        # Extract Katarina data
        kat_data= extract_katarina_match_data(match_data, puuid)
        
        if kat_data:
            all_matches_data.append(kat_data)
            katarina_count += 1
            result= "WIN" if kat_data['win'] == 1 else "LOSS"
            kda= f"{kat_data['kills']}/{kat_data['deaths']}/{kat_data['assists']}"
            print(f"Match {i+1}/{len(all_match_ids)}: Katarina {result} | KDA: {kda} | Total Kat games: {katarina_count}")
        else:
            print(f"Match {i+1}/{len(all_match_ids)}: Skipped (not Katarina CLASSIC)")
        
        # Rate limiting
        time.sleep(1.5)
    
    # Convert to DataFrame
    if not all_matches_data:
        print("\n No Katarina games found!")
        return None
    
    df= pd.DataFrame(all_matches_data)
    
    print(f"\n {'='*60}")
    print(f"DATA COLLECTION COMPLETE!")
    print('='*60)
    print(f"   Total matches checked: {len(all_match_ids)}")
    print(f"   Katarina CLASSIC games found: {len(df)}")
    print(f"   Features collected: {len(df.columns)}")
    print(f"   Dataset shape: {df.shape}")
    
    return df

### Collect the Data & Save in CSV

In [21]:
# Collect 1000 matches (will make 10 API requests for match IDs)

katarina_df= collect_katarina_dataset(my_player['puuid'], num_matches= 1000)

# Display results
if katarina_df is not None:
    print(f"\n {'='*60}")
    print(f"DATASET PREVIEW:")
    print('='*60)
    print(katarina_df.head(10))
    
    print(f"\n {'='*60}")
    print(f"DATASET INFO:")
    print('='*60)
    print(katarina_df.info())
    
    print(f"\n {'='*60}")
    print(f"BASIC STATISTICS:")
    print('='*60)
    print(katarina_df.describe())
    
    # Save to CSV
    katarina_classic= 'katarina_matches_1000.csv'
    katarina_df.to_csv(katarina_classic, index= False)
    print(f"\n Dataset saved to: {katarina_classic}")


COLLECTING KATARINA DATASET
Target: Check last 1000 matches...
Will make 10 API requests (batches of 100)

Batch 1/10: Fetching matches 0 to 99...
Retrieved 100 match IDs (Total so far: 100)
Batch 2/10: Fetching matches 100 to 199...
Retrieved 100 match IDs (Total so far: 200)
Batch 3/10: Fetching matches 200 to 299...
Retrieved 100 match IDs (Total so far: 300)
Batch 4/10: Fetching matches 300 to 399...
Retrieved 100 match IDs (Total so far: 400)
Batch 5/10: Fetching matches 400 to 499...
Retrieved 100 match IDs (Total so far: 500)
Batch 6/10: Fetching matches 500 to 599...
Retrieved 100 match IDs (Total so far: 600)
Batch 7/10: Fetching matches 600 to 699...
Retrieved 100 match IDs (Total so far: 700)
Batch 8/10: Fetching matches 700 to 799...
Retrieved 100 match IDs (Total so far: 800)
Batch 9/10: Fetching matches 800 to 899...
Retrieved 100 match IDs (Total so far: 900)
Batch 10/10: Fetching matches 900 to 999...
Retrieved 100 match IDs (Total so far: 1000)

 Total match IDs retri

In [22]:
df_kata= pd.read_csv(katarina_classic)
df_kata

,game_duration,win,kills,deaths,assists,cs,gold_earned,damage_to_champions,damage_taken,level,...,team_barons,team_towers,first_blood,kda_ratio,cs_per_min,gold_per_min,damage_per_min,kill_participation,kill_share,death_share
0,1978,0,11,5,2,236,15778,26594,24002,18,...,1,7,0,2.60,7.16,478.60,806.69,0.406,0.344,0.161
1,1725,0,7,3,1,203,11595,16284,18857,16,...,1,3,1,2.67,7.06,403.30,566.40,0.276,0.241,0.073
2,2020,1,11,2,8,148,13404,24535,21070,17,...,1,8,1,9.50,4.40,398.14,728.76,0.487,0.282,0.091
3,944,0,1,6,0,59,3971,4494,12750,9,...,0,0,1,0.17,3.75,252.39,285.64,0.333,0.333,0.333
4,1670,0,1,3,4,174,8590,7802,13334,15,...,0,1,0,1.67,6.25,308.62,280.31,0.333,0.067,0.088
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
455,1544,0,6,8,2,108,8339,16536,23204,12,...,0,2,0,1.00,4.20,324.05,642.59,0.258,0.194,0.205
456,1648,1,22,0,7,173,15854,32460,17529,17,...,0,9,1,29.00,6.30,577.21,1181.80,0.644,0.489,0.000
457,2217,0,13,4,8,212,16518,33320,30088,17,...,0,4,1,5.25,5.74,447.04,901.76,0.553,0.342,0.095
458,1645,0,13,4,1,197,14540,23854,24872,15,...,0,3,0,3.50,7.19,530.33,870.05,0.500,0.464,0.121


In [23]:
df_kata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 460 entries, 0 to 459
Data columns (total 21 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   game_duration        460 non-null    int64  
 1   win                  460 non-null    int64  
 2   kills                460 non-null    int64  
 3   deaths               460 non-null    int64  
 4   assists              460 non-null    int64  
 5   cs                   460 non-null    int64  
 6   gold_earned          460 non-null    int64  
 7   damage_to_champions  460 non-null    int64  
 8   damage_taken         460 non-null    int64  
 9   level                460 non-null    int64  
 10  team_dragons         460 non-null    int64  
 11  team_barons          460 non-null    int64  
 12  team_towers          460 non-null    int64  
 13  first_blood          460 non-null    int64  
 14  kda_ratio            460 non-null    float64
 15  cs_per_min           460 non-null    flo